In [4]:
import geopandas as gpd

# 1. Load the polygons
polygons = gpd.read_file(r"D:\landslide\tehri_landslide\data\GeospatialMapsPackage\Landslide Polygon.shp")

# 2. Print columns to find the ID name
print("Your available columns are:", polygons.columns.tolist())

Your available columns are: ['OBJECTID', 'LULC', 'GEOM', 'STATE', 'DISTRICT', 'TOPOSHEET', 'MATERIAL_T', 'LANDSLIDE_', 'CENT_X', 'CENT_Y', 'DATA_TYPE', 'geometry']


# New dataset

In [11]:
import rasterio
import geopandas as gpd
import pandas as pd
import numpy as np
import os
from shapely.geometry import Point

# ==========================================
# 1. SETUP PATHS
# ==========================================
tehri_path = r"F:\New folder (2)\tehri_new\tehri.shp"
poly_path = r"D:\landslide\tehri_landslide\data\GeospatialMapsPackage\Landslide Polygon.shp"
output_csv = r"D:\landslide\final_data\Tehri_Clean_Centroids.csv"

rasters = [
    r"D:\landslide\final_data\Elevation_10m.tif",
    r"D:\landslide\final_data\Slope_deg_10m.tif",
    r"D:\landslide\final_data\Aspect_deg_10m.tif",
    r"D:\landslide\final_data\Curvature_10m.tif",
    r"D:\landslide\final_data\TRI_10m.tif",
    r"D:\landslide\final_data\TWI_clean_10m.tif",
    r"D:\landslide\final_data\Rainfall_25yr_Mean_UTM_32644.tif",
    r"D:\landslide\final_data\distance_streams_tehri.tif",
    r"D:\landslide\final_data\distance_roads_tehri.tif",
    r"D:\landslide\final_data\distance_faults_tehri.tif",
    r"D:\landslide\final_data\NDVI_Tehri_10m.tif",
    r"D:\landslide\final_data\BSI_lite_Tehri_10m.tif",
    r"D:\landslide\final_data\Soil_Tehri_10m.tif",
    r"D:\landslide\final_data\lithology_tehri_10m.tif",
    r"D:\landslide\final_data\lulc_tehri_10m.tif",
    r"D:\landslide\final_data\Geomorphon_clean_10m.tif"
]

# ==========================================
# 2. LOAD BOUNDARIES
# ==========================================
tehri = gpd.read_file(tehri_path).to_crs("EPSG:32644")
all_polys = gpd.read_file(poly_path).to_crs("EPSG:32644")
tehri_polys = gpd.sjoin(all_polys, tehri, how="inner", predicate="intersects")

# ==========================================
# 3. EXTRACTION WITH NODATA FILTER
# ==========================================
# We define our "Bad" values to check against
invalid_values = [-9999, -3.4028235e+38, 9999, -4000]

final_data = []

# Open all rasters at once for efficient sampling
srcs = [rasterio.open(r) for r in rasters]
feat_names = [os.path.basename(r).replace("_10m.tif", "").replace(".tif", "") for r in rasters]

print("Starting clean extraction...")

# --- Process Landslides ---
for _, row in tehri_polys.iterrows():
    c = row.geometry.centroid
    point_data = {'x': c.x, 'y': c.y, 'landslide_id': row['OBJECTID'], 'label': 1}
    
    is_valid = True
    for i, src in enumerate(srcs):
        val = list(src.sample([(c.x, c.y)]))[0][0]
        # Check if value is NoData or physically impossible (like -3980 slope)
        if val in invalid_values or (feat_names[i] == 'Slope_deg' and val < 0):
            is_valid = False
            break
        point_data[feat_names[i]] = val
    
    if is_valid:
        final_data.append(point_data)

# --- Process Safe Zones (Balanced) ---
num_ls = len(final_data)
safe_zone = tehri.geometry.unary_union.difference(tehri_polys.geometry.unary_union)
minx, miny, maxx, maxy = safe_zone.bounds

print(f"Sampling {num_ls} clean safe points...")
while len([d for d in final_data if d['label'] == 0]) < num_ls:
    p = Point(np.random.uniform(minx, maxx), np.random.uniform(miny, maxy))
    if p.within(safe_zone):
        point_data = {'x': p.x, 'y': p.y, 'landslide_id': -1, 'label': 0}
        is_valid = True
        for i, src in enumerate(srcs):
            val = list(src.sample([(p.x, p.y)]))[0][0]
            if val in invalid_values or (feat_names[i] == 'Slope_deg' and val < 0):
                is_valid = False
                break
            point_data[feat_names[i]] = val
        
        if is_valid:
            final_data.append(point_data)

# Close rasters
for s in srcs: s.close()

# Save
df_final = pd.DataFrame(final_data)
df_final.to_csv(output_csv, index=False)
print(f"CLEAN CSV CREATED: {len(df_final)} balanced rows.")

Starting clean extraction...


C:\Users\sangh\AppData\Local\Temp\ipykernel_22128\2118397490.py:74: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  safe_zone = tehri.geometry.unary_union.difference(tehri_polys.geometry.unary_union)


Sampling 2591 clean safe points...
CLEAN CSV CREATED: 5182 balanced rows.


In [13]:
import pandas as pd
import numpy as np

# 1. LOAD DATA
file_path = r"Tehri_Landslide.csv"
df = pd.read_csv(file_path)

# 2. FILL NaNs (Imputation)
# We use the median to preserve the distribution of the 10m topographic data
df['TWI_clean'] = df['TWI_clean'].fillna(df['TWI_clean'].median())

# 3. FIX COLUMN NAMES
# Renaming to professional, short titles for easier coding and better charts
rename_dict = {
    'Elevation': 'Elevation',
    'Slope_deg': 'Slope',
    'Aspect_deg': 'Aspect',
    'Curvature': 'Curvature',
    'TRI': 'TRI',
    'TWI_clean': 'TWI',
    'Rainfall_25yr_Mean_UTM_32644': 'Rainfall',
    'distance_streams_tehri': 'Dist_Streams',
    'distance_roads_tehri': 'Dist_Roads',
    'distance_faults_tehri': 'Dist_Faults',
    'NDVI_Tehri': 'NDVI',
    'BSI_lite_Tehri': 'BSI',
    'Soil_Tehri': 'Soil',
    'lithology_tehri': 'Lithology',
    'lulc_tehri': 'LULC',
    'Geomorphon_clean': 'Geomorphon'
}

df = df.rename(columns=rename_dict)

# 4. FINAL VERIFICATION
print("--- Post-Fix Dataset Audit ---")
print(f"Total Rows: {len(df)}")
print(f"Total NaNs remaining: {df.isna().sum().sum()}")
print("\nFinal Column Names:")
print(df.columns.tolist())

# 5. SAVE THE POLISHED DATASET
output_path = r"D:\landslide\Tehri_Landslide_Data.csv"
df.to_csv(output_path, index=False)
print(f"\n Polished dataset saved to: {output_path}")

--- Post-Fix Dataset Audit ---
Total Rows: 5182
Total NaNs remaining: 0

Final Column Names:
['x', 'y', 'landslide_id', 'label', 'Elevation', 'Slope', 'Aspect', 'Curvature', 'TRI', 'TWI', 'Rainfall', 'Dist_Streams', 'Dist_Roads', 'Dist_Faults', 'NDVI', 'BSI', 'Soil', 'Lithology', 'LULC', 'Geomorphon']

 Polished dataset saved to: D:\landslide\Tehri_Landslide_Data.csv


In [14]:
df = pd.read_csv("Tehri_Landslide_Data.csv")

In [15]:
df.columns.tolist()

['x',
 'y',
 'landslide_id',
 'label',
 'Elevation',
 'Slope',
 'Aspect',
 'Curvature',
 'TRI',
 'TWI',
 'Rainfall',
 'Dist_Streams',
 'Dist_Roads',
 'Dist_Faults',
 'NDVI',
 'BSI',
 'Soil',
 'Lithology',
 'LULC',
 'Geomorphon']

In [16]:
df.isna().sum()

x               0
y               0
landslide_id    0
label           0
Elevation       0
Slope           0
Aspect          0
Curvature       0
TRI             0
TWI             0
Rainfall        0
Dist_Streams    0
Dist_Roads      0
Dist_Faults     0
NDVI            0
BSI             0
Soil            0
Lithology       0
LULC            0
Geomorphon      0
dtype: int64